In [ ]:
import gc
import pandas as pd
import fastparquet
from pathlib import Path
%matplotlib inline
%load_ext autoreload
%autoreload 2
from imports import *
import scipy.io
from config import dir_config, ephys_config
from io import BytesIO
from pptx import Presentation
from pptx.util import Inches

compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

In [ ]:
session_metadata = pd.read_csv(processed_dir / "sessions_metadata.csv")

In [ ]:
def downsample_and_guassian_smooth(signal, original_rate=30000, target_rate=1000, sigma_ms=5):
    # downsample with mean pooling
    n_samples = signal.shape[0] // (original_rate // target_rate)
    downsampled = np.nan * np.ones(n_samples)
    for i in range(n_samples):
        downsampled[i] = np.nanmean(signal[i*(original_rate//target_rate):(i+1)*(original_rate//target_rate)])
    
    # gaussian smooth
    sigma_samples = int(sigma_ms * target_rate / 1000)  # convert ms to samples
    smoothed = scipy.ndimage.gaussian_filter1d(downsampled, sigma=sigma_samples)
    
    return smoothed

## Loop over all sessions

In [ ]:
prs = Presentation()
prs.slide_width = Inches(18)
prs.slide_height = Inches(6)

window_length = 10501

for _, session_row in session_metadata.iterrows():
    session_id = session_row.session_id

    eye_parquet = compiled_dir / session_id / f"{session_id}_eye_tracking_processed.parquet"
    timestamp_filename = compiled_dir / session_id / f"{session_id}_timestamps_cleaned.csv"

    if not eye_parquet.exists() or not timestamp_filename.exists():
        print(f"Skipping {session_id}: missing files")
        continue

    print(f"Processing {session_id}...")
    timestamps = pd.read_csv(timestamp_filename)
    saccades = timestamps.response_onset.dropna().values

    pf = fastparquet.ParquetFile(str(eye_parquet))
    n_ts = sum(rg.num_rows for rg in pf.row_groups)

    # Read just the first timestamp to compute row indices.
    # Assumes uniform 30 kHz sampling: row_index = timestamp - first_ts.
    first_chunk = next(pf.iter_row_groups(columns=["timestamp"]))
    first_ts = int(first_chunk.timestamp.iloc[0])
    del first_chunk
    gc.collect()

    starts = np.clip((saccades - 3000 - first_ts).astype(int), 0, n_ts - window_length)
    win_ends = starts + window_length

    # Preallocate window matrices as float32 (~56 MB each for ~1344 trials)
    wx = np.full((len(starts), window_length), np.nan, dtype=np.float32)
    wy = np.full((len(starts), window_length), np.nan, dtype=np.float32)

    # Stream through the file one row group at a time -- never load the full file
    row_offset = 0
    for chunk in pf.iter_row_groups(columns=["eye_x", "eye_y"]):
        rg_size = len(chunk)
        rg_end = row_offset + rg_size

        overlap_mask = (starts < rg_end) & (win_ends > row_offset)
        if overlap_mask.any():
            ex_rg = chunk.eye_x.values
            ey_rg = chunk.eye_y.values
            for sac_i in np.where(overlap_mask)[0]:
                ov_s = max(starts[sac_i], row_offset)
                ov_e = min(win_ends[sac_i], rg_end)
                dst_s, dst_e = ov_s - starts[sac_i], ov_e - starts[sac_i]
                src_s, src_e = ov_s - row_offset, ov_e - row_offset
                wx[sac_i, dst_s:dst_e] = ex_rg[src_s:src_e]
                wy[sac_i, dst_s:dst_e] = ey_rg[src_s:src_e]

        del chunk
        row_offset += rg_size
    gc.collect()

    # Compute radial distance, then downsample and smooth
    eye_data = np.sqrt(wx**2 + wy**2)
    del wx, wy
    gc.collect()

    eye_data_processed = np.nan * np.ones((eye_data.shape[0], eye_data.shape[1] // 30))
    for i in range(eye_data.shape[0]):
        eye_data_processed[i, :] = downsample_and_guassian_smooth(eye_data[i, :], sigma_ms=5)
    eye_data_processed = eye_data_processed.T
    del eye_data
    gc.collect()

    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(session_id, fontsize=14)

    axs[0].plot(eye_data_processed, color="blue", alpha=0.2)
    axs[0].axvline(101, color="k", linestyle="--", label="saccade onset")
    axs[0].set_ylabel("Position")

    axs[1].plot(1000 * np.diff(eye_data_processed, axis=0), color="red", alpha=0.2)
    axs[1].axvline(100, color="k", linestyle="--", label="saccade onset")
    axs[1].set_ylabel("Velocity")

    axs[2].plot(1e6 * np.diff(np.diff(eye_data_processed, axis=0), axis=0), color="green", alpha=0.2)
    axs[2].axvline(99, color="k", linestyle="--", label="saccade onset")
    axs[2].set_ylabel("Acceleration")

    plt.tight_layout()
    plt.show()

    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    buf.seek(0)

    slide = prs.slides.add_slide(prs.slide_layouts[6])  # blank layout
    slide.shapes.add_picture(buf, Inches(0), Inches(0), width=prs.slide_width, height=prs.slide_height)

    plt.close(fig)

print("Done processing all sessions.")

In [ ]:
output_path = processed_dir / "saccade_onset_verification_cleaned.pptx"
prs.save(output_path)
print(f"Saved PPT to: {output_path}")